# 🤖 PhysiSim AI — Google Colab GPU Backend (Zero-CAPEX Stack)

Nền tảng giả lập Physical AI / Embodied AI chạy trên **GPU T4 Google Colab (0 VNĐ)**.

### 🛡️ Đã tích hợp giải pháp chống mất dữ liệu & Tunnel Miễn Phí:
1. **Google Drive Persistent Mount**: Tự động lưu toàn bộ dataset HDF5 và checkpoints vào Google Drive (`/MyDrive/PhysiSim_AI/`). Dù Colab có timeout hay restart, **dữ liệu vẫn còn nguyên 100%**.
2. **Cloudflare Quick Tunnel (Mặc định - 0 cần đăng ký, 0 cần Token)** hoặc **ngrok**.
3. **Hugging Face Hub Auto-Sync**: Tự động đồng bộ các file `.h5` lên Hugging Face Hub Dataset ngay khi xuất dữ liệu.

---
### 🚀 Hướng dẫn nhanh:
1. **Menu Runtime** → **Change runtime type** → Chọn **GPU (T4)** → Bấm **Save**.
2. Bấm **Runtime** → **Run all** (hoặc chạy lần lượt các Cell bên dưới).
3. Cho phép truy cập Google Drive khi popup xuất hiện.
4. Copy đường link **PUBLIC URL** ở Cell 4 dán vào ô URL trên Web UI PhysiSim AI.

In [ ]:
# =========================================================
# 1. GẮN GOOGLE DRIVE LƯU TRỮ VĨNH VIỄN (PERSISTENT STORAGE)
# =========================================================
import os
from google.colab import drive

# Gắn Google Drive cá nhân
drive.mount('/content/drive')

# Thư mục lưu trữ vĩnh viễn trên Drive
DRIVE_BASE_DIR = "/content/drive/MyDrive/PhysiSim_AI"
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, "datasets")
MODEL_DIR = os.path.join(DRIVE_BASE_DIR, "models")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Thiết lập biến môi trường để Backend tự động lưu vào Drive
os.environ["PHYSISIM_DATA_DIR"] = DATASET_DIR
os.environ["PHYSISIM_MODEL_DIR"] = MODEL_DIR

print("=" * 60)
print("✅ GOOGLE DRIVE PERSISTENT STORAGE SẴN SÀNG!")
print(f"📁 Thư mục lưu Dataset : {DATASET_DIR}")
print(f"📁 Thư mục lưu Model   : {MODEL_DIR}")
print("=" * 60)

In [ ]:
# =========================================================
# 2. KIỂM TRA GPU NVIDIA T4 & CÀI ĐẶT THƯ VIỆN
# =========================================================
!nvidia-smi

import torch
print(f"🔥 PyTorch CUDA Sẵn Sàng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Tên GPU: {torch.cuda.get_device_name(0)}")

# Cài đặt các gói phụ thuộc
!pip install --quiet fastapi uvicorn pyngrok nest_asyncio mujoco h5py huggingface_hub pydantic numpy pillow requests

In [ ]:
# =========================================================
# 3. CẤU HÌNH HUGGING FACE HUB AUTO-SYNC (TÙY CHỌN)
# =========================================================
# Nhập Token và Repo ID từ Hugging Face (https://huggingface.co/settings/tokens)
HF_TOKEN = ""       # Ví dụ: "hf_xxxxxxxxxxxxxxxxxxxx"
HF_REPO_ID = ""     # Ví dụ: "username/physisim-robot-dataset"

if HF_TOKEN and HF_REPO_ID:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HF_REPO_ID"] = HF_REPO_ID
    from huggingface_hub import HfApi
    try:
        api = HfApi()
        api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", token=HF_TOKEN, exist_ok=True)
        print(f"✅ Đã kết nối Hugging Face Hub Dataset: https://huggingface.co/datasets/{HF_REPO_ID}")
    except Exception as e:
        print(f"⚠ Thông báo kết nối HF: {e}")
else:
    print("ℹ Bạn chưa nhập HF Token (bạn có thể nhập trực tiếp từ Web UI khi chạy)")

In [ ]:
# =========================================================
# 4. KHỞI CHẠY BACKEND (CLOUDFLARE QUICK TUNNEL / NGROK)
# =========================================================
import os
import sys
import subprocess
import time
import re
import nest_asyncio
import uvicorn

nest_asyncio.apply()

# Nếu bạn có ngrok token, điền vào đây; Nếu ĐỂ TRỐNG hệ thống sẽ tự dùng Cloudflare Tunnel (Miễn phí 100%, 0 cần token)
NGROK_AUTH_TOKEN = ""

public_url = None

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok.kill()
    public_url = ngrok.connect(8000).public_url
else:
    print("⚡ Đang khởi tạo Cloudflare Quick Tunnel (Miễn phí, không cần tài khoản/token)...\n")
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    for _ in range(50):
        line = cf_proc.stderr.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
        time.sleep(0.1)

print("=" * 65)
print("🎉 PHYSISIM AI GPU BACKEND ĐÃ SẴN SÀNG!")
print(f"🔗 PUBLIC URL: {public_url}")
print("👉 HÃY COPY ĐƯỜNG LINK TRÊN VÀ DÁN VÀO Ô URL TRÊN WEB UI PHYSISIM AI")
print("=" * 65)

# Khởi động server FastAPI
from app.main import app
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

### 💡 Mẹo: Chống Tự Động Ngắt Tab Colab (Anti-Idle):
Mở **Developer Console** (`F12` hoặc `Cmd+Opt+I`) trên tab Colab và dán đoạn mã sau vào Console:
```javascript
setInterval(() => {
  document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click();
  console.log("PhysiSim Colab Keep-Alive Ping ✅");
}, 60000);
```